## LECTURE 4: Stack & Procedures -- The #1 Exam Topic
   This lecture is the linchpin of Part II. Every past paper has a question
   where you're given assembly and need to reconstruct the C function. To do 
   that, you need to understand four things deeply: the stack, `call`/`ret`,
   the calling convention, and register saving. ...


1. The Stack -- A Region of Memory
   The stack is just a chunk of memory that grows DOWNWARD (toward lower 
   addresses). `%rsp` always points to the "top" of the stack -- but since it
   grows downwards, "top" means the lowest address currently in use.

   Two fundamental operations:

   `pushq %reg` does two things atomically: (1) decrement `%rsp` by 8, then (2)
   write the register's value to the memory at the new `%rsp`. It's equivalent
   to `subq $8, %rsp` followed by `movq %reg, (%rsp)`.

   `popq %reg` is the reverse: (1) read the value at `(%rsp)` into the register,
   then (2) increment `%rsp` by 8. Equivalent to `movq (%rsp), %reg` followed by
   `addq $8, %rsp`.

   Why 8? because in x86-64, addresses are 4 bits = 8 bytes. The stack is 
   primarily used to store addresses (return addresses, saved registers), so
   everything is aligned to 8 bytes.


2. call and ret -- How Functions Transfer Control
   `call label` does two things: (1) push the address of the next instruction
   (the return address) onto the stack, then (2) jump to `label`. So if you have
   :

```asm
40050a:  callq 400502 <add>
40050f:  mov %eax, (%rbx)
```
   The `call` pushes `0x40050f` onto the stack (the address of the `mov`
   instrcution), then jumps to `0x400502`.

   `ret` is the reverse: pop the top of the stack into `%rip` (the program 
   counter). So it reads `0x40050f` from the stack, increments `%rsp` by 8, and 
   jumps to that address. Execution continues with the `mov` instrcution. 

   This is exactly like the JAL/JALR mechanism from RISC-V part I, except RISC-V
   stores the return address in a register (`ra`/`x1`) while x8 stores it on the
   stack. The stack approach is more powerful becuase it naturally supports 
   arbitrary nesting -- each `call` pushes another return address, and each `ret`
   pops the right one. 


3. The Calling Convention -- The Contract between Caller and Callee
   
   ...

   1 - %rdi
   2 - %rsi
   3 - %rdx
   4 - %rcx
   5 - %r8
   6 - %r9

   Any additional arguments (7th, 8th, ...) are pushed onto the stack in reverse
   order. The return value goes in `%rax`.

   MEMORISE THIS ORDER. When you see assembly and need to figure out what the
   function's parameters are, the very first thing you do is look at which of 
   these registers the function uses without writing to them first. If the
   first instruction is `addl %edi, %esi`, you immediately know this function
   takes at least two `int` arguments.

   The register naming also tells you the types: `%edi` (32-bit `e` prefix) 
   means `int`, `%rdi` (64-bit `r` prefix) means `long` or a pointer, ...
   `di` means short:16-bit... `%dil` means char:8-bit...


4. REGISTER SAVING -- Caller-Saved vs Callee-Saved
   Here's the problem: caller and callee share the same 16 registers. If `foo`
   has a value in `%rdx` that it needs after calling `bar`,    

   ...

2. Caller-Saved vs. Callee-Saved Registers
   
   Think of the CPU's 16 registers as a shared whiteboard. When Function A (the
   Caller) temporarily pauses to let Function B (the Callee) run, they have to
   agree on who cleans up the whiteboard.
      - Caller-Saved ("Trashable"): These are scratchpads. Function B is allowed
        to overwrite them immediately. If Function A needs the data in these 
        registers after Function B finishes, Function A better save it somewhere
        else first. 
         * Includes: `%rax`, `%rdi`, `%rsi`, `%rdx`, `%rcx`, `%r8`, `%r9`, `%r10`, `%r11`
      - Callee-Saved ("Sacred"): Function B is allowed to use these, but it MUST
        take a picture of what was there, use the space, and then erase and 
        redraw the original data before returning to Function A.
         * Includes: `%rbx`, `%rbp`, `%r12`-`%r15`


---
The Ultimate Exam Example
   ...: stashing a parameter into a callee-saved register to survive the 
   function call.


THE C CODE:
```c
long calculate_total(long x) {
   long y = helper_func(x);
   return x + y;              // We need `x` AFTER the function call!
}
```
   
   THE PROBLEM: When `calculate_total` starts, `x` is living in `%rdi`. To call
   `helper_func(x)`, we leave `x` in `%rdi`. But `helper_func` is allowed to 
   completely trash `%rdi` while it runs! How do we keep `x` safe so we can add
   it to `y` later?


THE ASSEMBLY SOLUTION:
```asm
calculate_total:
    pushq %rbx       # 1. We want to use %rbx to keep `x` safe.
                     #    Since %rbx is callee-saved, WE must save the old value first

    movq %rdi, %rbx  # 2. Stash `x` safely into %rbx
    call helper_func # 3. Call the helper. %rdi might get destroyed here. The 
                     #    result `y` comes back in %rax.

    addq %rbx, %rax  # 4. Add our safely stored `x` (%rbx) to `y` (%rax). %rax
                     #    now holds the final return value.

    popq %rbx        # 5. Restore the original %rbx we saved in step 1. 
    ret              # 6. Return to whoever called us. 
```

   WHY THIS MATTERS: When you are reading assembly in an exam and see 
   `pushq %rbx` followed immediately by `movq %rdi, %rbx`, you instantly know 
   the logic: "Ah, this function is going to call another function, and it needs
   to remember the first argument for later."

```asm
apply_discount:
    pushq %rbx
    movq  %rdi, %rbx
    call  calculate_discount

    # movq  %rbx, %rsi
    # subq  %rax, %rsi
    # movq  %rsi, %rax
    subq  %rbx, %rax
    negq  %rax

    popq  %rbx
    ret   
```

```asm
double_and_add:
    pushq %rsi
    call  double_and_add
    popq  %rdx
    addq  %rdx, %rax
    ret
```

```asm
compute_combo:  
    leaq  (%edi, %esi), %ecx
    pushq %ebx
    movq  %ecx, %ebx
    call  multiply_them 
    addq  %ebx, %eax
    popq  %ebx
    ret
```    

```asm
count_up:     
    movq  $0, %eax
    jmp   .L_test
.L_loop:    
    incl  %rax
.L_test:
    cmpq  %rdi, %rax
    jl    .L_loop
    ret
```

```asm
sum_down:       // %edi = n // %rax = sum
    movl  $0, %rax
.L_loop:
    addq  %rdi, %rax
    decl  %rdi
.L_test:
    cmpq  $0, $rdi
    jg    .L_loop
    ret
```

---

-- ... Those things with the colons at the end are called LABELS.

   Think of a label as a "bookmark" in your code. It doesn't actually do 
   anything on its own; it just gives a name to a specific memory address so 
   that your `jmp` (jump) instructions know where to go.

   Here is the breakdown of the conventions for naming these labels, 
   specifically the `.` and the `L`.


1. THE DOT (`.`): Local vs. Global
   In the GNU Assembler (the standard for Linux/AT&T syntax), the dot `.` at the 
   start of a label means it is a LOCAL SYMBOL.
      * GLOBAL LABELS (No Dot): Labels like `main:` or `calculate_total:` are 
        global. This means the Linker can see them, and other C files or
        assembly files can call them.
      * LOCAL LABELS (With Dot): Labels like `.loop:` or `.continue:` are local.
        They are "invisible" outside of the current file.


WHY IS THE DOT SO IMPORTANT?
   Imagine you write two different functions in the same file, and both of them
   have a while loop If you just named the label `loop:`

-- In x86-64 assembly language, `%rip` refers to the INSTRCUTION POINTER 
   REGISTER (or program counter), which holds the memory address of the next 
   instruction to be executed by the CPU. While it is a specialised 64-bit 
   register, its most common appearance in modern assembly code is in 
   RIP-relative addressing, which is used to access global variables and data. 

-- ... JAL (Jump and Link) is an instruction used to call functions and 
   subroutines. It performs two key actions: it jumps to a target address and
   saves the address of the next instruction (return address) into a register
   (ra), allowing the program to return later. 
        (RISC-V and MIPS)   